# Prueba de servicios
Hay que correr las celdas en orden. Los servicios tienen que estar levantados con `docker compose --profile all up -d` o desde la aplicacion de docker desktop.

## 1. Se configuran las distintas conexiones 

In [11]:
import os
import io
import boto3
from sklearn.datasets import load_iris
import pandas as pd

In [12]:
os.environ["AWS_ACCESS_KEY_ID"] = "minio"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"
os.environ["AWS_ENDPOINT_URL_S3"] = "http://localhost:9000"     
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"  
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

MLFLOW_URL = "http://localhost:5001"

## 2. Subir un archivo al bucket `data` de juguete.

In [13]:
df = load_iris(as_frame=True).frame
print(df.shape)
df.head()

(150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [14]:
s3 = boto3.client("s3")

buffer = io.StringIO()
df.to_csv(buffer, index=False)
s3.put_object(Bucket="data", Key="prueba/iris.csv", Body=buffer.getvalue())

# Listar lo que hay en el bucket
for obj in s3.list_objects_v2(Bucket="data").get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

prueba/iris.csv 2928 bytes
prueba/resumen.csv 218 bytes


## 3. Leerlo de vuelta
Comprobamos que lo que está en el bucket es lo mismo que subimos.

In [15]:
respuesta = s3.get_object(Bucket="data", Key="prueba/iris.csv")
df_leido = pd.read_csv(respuesta["Body"])
print(df_leido.shape)
df_leido.head()

(150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## 4. Entrenar un modelo y registrarlo en MLflow
El entrenamiento corre **localmente**. MLflow guarda parámetros y métricas en Postgres, y el modelo en el bucket `mlflow`.

In [16]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(MLFLOW_URL)
mlflow.set_experiment("prueba")

X = df_leido.drop(columns="target")
y = df_leido["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

for C in [0.01, 0.1, 1.0]:
    with mlflow.start_run(run_name=f"logreg_C={C}"):
        modelo = LogisticRegression(C=C, max_iter=500).fit(X_train, y_train)
        acc = accuracy_score(y_test, modelo.predict(X_test))

        mlflow.log_param("C", C)
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(modelo, name="modelo")
        print(f"C={C}  accuracy={acc:.3f}")

C=0.01  accuracy=0.822
🏃 View run logreg_C=0.01 at: http://localhost:5001/#/experiments/1/runs/fa1fb2b607d84f11ad62510d9687ead5
🧪 View experiment at: http://localhost:5001/#/experiments/1
C=0.1  accuracy=0.956
🏃 View run logreg_C=0.1 at: http://localhost:5001/#/experiments/1/runs/1e646dff3d7947c3b6316ab75c850e86
🧪 View experiment at: http://localhost:5001/#/experiments/1
C=1.0  accuracy=1.000
🏃 View run logreg_C=1.0 at: http://localhost:5001/#/experiments/1/runs/c7e76ff4a011442cb289cebe412c5669
🧪 View experiment at: http://localhost:5001/#/experiments/1


## 5. Revisamos
- **RustFS** (http://localhost:9001): bucket `data` → `prueba/iris.csv`. Bucket `mlflow` → carpetas nuevas con los modelos.
- **MLflow** (http://localhost:5001): experimento `prueba` con 3 runs.